# BigQuery Timestamp Fetch (direct SQL)
Reads `data/input/seeds.csv` and retrieves block timestamps using direct BigQuery SQL queries.

In [ ]:
import pandas as pd
from google.cloud import bigquery

seeds = pd.read_csv('../data/input/seeds.csv', header=None)[0].tolist()
client = bigquery.Client()
sql = """
SELECT TO_HEX(`hash`) AS txid, block_timestamp
FROM `bigquery-public-data.crypto_bitcoin.transactions`
WHERE TO_HEX(`hash`) IN UNNEST(@txids)
"""
job_config = bigquery.QueryJobConfig(query_parameters=[bigquery.ArrayQueryParameter('txids','STRING',seeds)])
rows = client.query(sql, job_config=job_config).result()
timestamps = {row.txid: row.block_timestamp.isoformat() for row in rows}
pd.DataFrame.from_dict(timestamps, orient='index', columns=['timestamp']).head()